In [ ]:
# List of required packages
required_packages <- c( 
    "R.utils",
    "dplyr", 
    "tidyr",
    "data.table",
    "ggplot2",
    "lubridate",
    "broom",
    "scales",
    "patchwork",
    "glmnet"
)

# Function to check and install missing packages
install_and_load <- function(packages) {
  installed_packages <- rownames(installed.packages())
  for (pkg in packages) {
    if (!pkg %in% installed_packages) {
      install.packages(pkg, dependencies = TRUE)
    }
    suppressMessages(library(pkg, character.only = TRUE))
  }
}

# Install and load the required packages
install_and_load(required_packages)

In [ ]:
# Define the file name
file_names <- c(
    "ukb_gp_for_analysis.rds"
)

# Check if the file exists in the current directory
for (file in file_names) {
    if (!file.exists(file)) {
      message("File not found. Downloading from DNAnexus...")

      # Execute the dx download command
      system(paste("dx download project-XXXXXXXXXXXXXXXXXXXXXXXX:/path/to/scratch/", file, sep = ""))
    }
}

# Load the data
data_raw <- readRDS("ukb_gp_for_analysis.rds")

In [ ]:
y_var   <- "LDL1"

log_vars    <- c("LDL0")  # log() in model
num_vars    <- c("age_at_LDL0", "n_rx_days_365d", "n_clin_days_365d", "ldl_prs", "cad_prs")
factor_vars <- c("sex", "prior_statin", "prior_CAD")

# ---- helper: complete-case restriction respecting log() constraints ----
prep_complete_cases <- function(dt, y, log_vars, num_vars, factor_vars, log_y = FALSE) {
  vars_needed <- unique(c(y, log_vars, num_vars, factor_vars))
  keep <- stats::complete.cases(dt[, ..vars_needed])

  # log() covariates require strictly positive values
  for (v in log_vars) keep <- keep & is.finite(dt[[v]]) & dt[[v]] > 0

  # outcome constraints depend on whether it's logged
  keep <- keep & is.finite(dt[[y]])
  if (isTRUE(log_y)) keep <- keep & dt[[y]] > 0

  dt[keep]
}

data <- prep_complete_cases(data_raw, y_var, log_vars, num_vars, factor_vars, log_y = TRUE)

In [ ]:
# ---- build formula programmatically ----
log_terms    <- sprintf("I(log(%s))", log_vars)
num_terms    <- num_vars
factor_terms <- sprintf("factor(%s)", factor_vars)

In [ ]:
# Ensure binary A
data <- data %>%
  mutate(
    A = as.integer(A),
    M = as.integer(has_LDL1 == TRUE)
  )
stopifnot(all(data$A %in% c(0L, 1L)))

# ----------------------------
# Denominator model: P(A=1 | X) via cv.glmnet
# ----------------------------
rhs_terms <- c(log_terms, num_terms, factor_terms, "ldl_prs", "cad_prs")
form <- as.formula(paste0("A ~ ", paste(rhs_terms, collapse = " + ")))

# Build model matrix for glmnet; drop intercept (glmnet adds it by default)
mm <- model.matrix(form, data = data)
y  <- model.response(model.frame(form, data = data))

x <- mm[, colnames(mm) != "(Intercept)", drop = FALSE]

# Fit cross-validated penalised logistic regression
mA_den <- cv.glmnet(
  x = x, y = y,
  family = "binomial",
  type.measure = "deviance",
  alpha = 1
)
coef(mA_den)

# Predict probabilities at lambda.min (or lambda.1se for more shrinkage)
pA1_den <- as.numeric(predict(mA_den, newx = x, s = "lambda.min", type = "response"))
pA1_den <- pmin(pmax(pA1_den, 1e-6), 1 - 1e-6)  # avoid 0/1

pA0_den <- 1 - pA1_den

# ----------------------------
# Stabilised weights
# ----------------------------
pA1_num <- mean(data$A == 1L, na.rm = TRUE)
pA0_num <- 1 - pA1_num

wA <- ifelse(data$A == 1L, pA1_num / pA1_den, pA0_num / pA0_den)

# Truncate weights
q05 <- quantile(wA, 0.05, na.rm = TRUE)
q95 <- quantile(wA, 0.95, na.rm = TRUE)
wA  <- pmin(pmax(wA, q05), q95)

# Attach to data
data <- data %>%
  mutate(
    pA1_den = pA1_den,
    pA_den  = ifelse(A == 1L, pA1_den, pA0_den),
    wA_ipw  = wA
  )

# Diagnostics
summary(data$wA_ipw)
quantile(data$wA_ipw, probs = c(0, 0.5, 0.9, 0.95, 0.99, 1), na.rm = TRUE)

In [ ]:
q0   <- as.numeric(quantile(data$LDL0, 0.00, na.rm = TRUE))
q33  <- as.numeric(quantile(data$LDL0, 0.33, na.rm = TRUE))
q66  <- as.numeric(quantile(data$LDL0, 0.66, na.rm = TRUE))
q100 <- as.numeric(quantile(data$LDL0, 1.00, na.rm = TRUE))
ldl0_cuts_strata <- c(q0, q33, q66, q100)

In [ ]:
fit_estimators_interval <- function(dat,
                                   ldl0_cuts = c(-Inf, Inf)) {
  stopifnot(length(ldl0_cuts) == 4)

  # Full cohort MSM/IPTW
  m_full <- lm(I(log(LDL1)) ~ age_at_LDL0 + sex + A + ldl_prs + A_x_ldl_prs,
              data = dat,
              weights = wA_ipw)
  out <- data.frame(
      estimator = "MSM via IPTW",
      estimate = coef(m_full)[["A_x_ldl_prs"]],
      se = summary(m_full)$coefficients["A_x_ldl_prs", "Std. Error"],
      lower_ci = confint(m_full)["A_x_ldl_prs", 1],
      upper_ci = confint(m_full)["A_x_ldl_prs", 2]
  )

  stratum_names <- c("MSM via IPTW (LDL-C 0-33%)",
                     "MSM via IPTW (LDL-C 34-66%)",
                     "MSM via IPTW (LDL-C 67-100%)")

  for (k in 1:3) {
    lo <- ldl0_cuts[k]
    hi <- ldl0_cuts[k + 1]

    if (k == 1) {
      S <- (dat$LDL0 >= lo) & (dat$LDL0 <= hi)
    } else {
      S <- (dat$LDL0 > lo) & (dat$LDL0 <= hi)
    }

    if (!any(S)) stop("Interval restriction kept 0 rows for stratum ", k, "; check cut points.")

    m_k <-   m_full <- lm(I(log(LDL1)) ~ age_at_LDL0 + sex + A + ldl_prs + A_x_ldl_prs,
              data = dat[S],
              weights = wA_ipw)
    out <- rbind(out, 
      data.frame(
        estimator = stratum_names[k],
        estimate = coef(m_k)[["A_x_ldl_prs"]],
        se = summary(m_k)$coefficients["A_x_ldl_prs", "Std. Error"],
        lower_ci = confint(m_k)["A_x_ldl_prs", 1],
        upper_ci = confint(m_k)["A_x_ldl_prs", 2]
      )                                 
    )
  }

  out
}

In [ ]:
out <- fit_estimators_interval(data, ldl0_cuts_strata)

In [ ]:
plot_dat <- out %>%
  mutate(
    estimator = factor(
      estimator,
      levels = c(
        "MSM via IPTW",
        "MSM via IPTW (LDL-C 0-33%)",
        "MSM via IPTW (LDL-C 34-66%)",
        "MSM via IPTW (LDL-C 67-100%)"
      )
    ),
    label = sprintf(
      "est=%.3f\n95%% CI\n[%.3f, %.3f]",
      estimate, lower_ci, upper_ci
    )
  ) %>%
  arrange(estimator)

# Self-contained colours
cols_strata <- c(
  "MSM via IPTW" = "#1B9E77",
  "MSM via IPTW (LDL-C 0-33%)" = "#E7298A",
  "MSM via IPTW (LDL-C 34-66%)" = "#E7298A",
  "MSM via IPTW (LDL-C 67-100%)" = "#E7298A"
)

alpha_map <- c(
  "MSM via IPTW" = 0.65,
  "MSM via IPTW (LDL-C 0-33%)" = 0.75,
  "MSM via IPTW (LDL-C 34-66%)" = 0.75,
  "MSM via IPTW (LDL-C 67-100%)" = 0.75
)

y_rng <- range(c(plot_dat$lower_ci, plot_dat$upper_ci), na.rm = TRUE)
y_lab <- max(plot_dat$upper_ci, na.rm = TRUE) + 0.08 * diff(y_rng)

plot_dat <- plot_dat %>%
  mutate(y = y_lab)

theme_pub <- theme_minimal(base_size = 12) +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    legend.title = element_blank(),
    legend.position = "top",
    plot.title.position = "plot",
    plot.title = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey25"),
    axis.title = element_text(face = "bold")
  )

p_box_strata <- ggplot(plot_dat, aes(x = estimator, y = estimate, colour = estimator)) +
  geom_hline(yintercept = 0, linetype = "dotted", linewidth = 0.8) +
  geom_errorbar(
    aes(ymin = lower_ci, ymax = upper_ci, alpha = estimator),
    width = 0.18,
    linewidth = 0.9,
    show.legend = FALSE
  ) +
  geom_point(
    aes(fill = estimator, alpha = estimator),
    size = 3.2,
    shape = 21,
    stroke = 0.8,
    colour = "black",
    show.legend = FALSE
  ) +
  geom_text(
    aes(y = y, label = label),
    colour = "black",
    size = 3.2,
    lineheight = 1.05,
    show.legend = FALSE
  ) +
  coord_cartesian(clip = "off") +
  scale_fill_manual(values = cols_strata) +
  scale_colour_manual(values = cols_strata) +
  scale_alpha_manual(values = alpha_map) +
  theme_pub +
  theme(
    legend.position = "none",
    axis.title.x = element_blank(),
    axis.text.x = element_text(angle = 18, hjust = 1),
    plot.margin = margin(8, 12, 8, 12)
  ) +
  labs(
    # title = "Effect on GxA estimate of baseline LDL-C stratification",
    # subtitle = "Frequentist point estimates with 95% confidence intervals.",
    y = "Effect modification by LDL PRS"
  )

p_box_strata
ggsave(
  plot = p_box_strata,
  width = 8,
  height = 6,
  dpi = 300,     
  filename = "statin_ldl_stratified_forest.png"
)